In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] ='0'
from huggingface_hub import notebook_login

- log in Hugging Face 

In [2]:
notebook_login()

# Phase 1: System Design & Stack Selection

- __Language & Framework:__ Python with LangChain. It provides a strong set of core components for building both RAG systems and tool-based agents.
- __Data Ingestion:__ WebBaseLoader for the SAGES and AME HTML links, and PyPDFLoader for the ERAS booklet.
- __Chunking Strategy:__ RecursiveCharacterTextSplitter. Since surgical steps and guidelines are highly sequential, overlapping chunks (e.g., 1000 tokens with a 200-token overlap) will help maintain context across operative steps.
- __Vector Store:__ ChromaDB. It is lightweight, run locally, and require zero external infrastructure, which is perfect for a CLI tool.
- __Agent Paradigm:__ A Tool-Calling Agent. I will wrap the RAG pipeline into a "Clinical Knowledge Retriever" tool. I will also implement Structured Output using Pydantic, requiring the LLM to format its response with specific fields (e.g., current_step, next_action, safety_warnings, etc) to satisfy the agent capability requirement.

### Here is a clean Python script using LangChain to ingest the web and PDF sources, along with a strategic chunking configuration.

# Phase 2: Data Ingestion & Chunking Strategy

In [3]:
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


### __Ingests clinical guidelines from URLs and PDFs, then chunks them for optimal Vector Store retrieval.__

## 1. Define the source documents

In [4]:
sages_url = "https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/"
ame_url = "https://ales.amegroups.org/article/view/5766/html"
eras_pdf_url = "https://www.urmc.rochester.edu/getmedia/c6bc9e17-c349-436c-926e-bf4f4e498d8d/ERAS-Cholecystectomy-Booklet.pdf"

### Load HTML Web Documents (SAGES & AME)

At a high level, *WebBaseLoader* in LangChain is a tool that handles the process of pulling HTML from a URL and turning it into a clean, usable document object.

I used it to load sources like the SAGES guidelines and the AME Operative Technique article. The advantage of *WebBaseLoader* is that it standardizes the output for the rest of the LangChain pipeline.

Most importantly, it automatically keeps the source URL as part of the document’s metadata. In a clinical workflow, knowing where the information comes from is important; if the agent suggests a specific surgical technique, we need to link that recommendation back to the exact SAGES source.

In [5]:
web_loader = WebBaseLoader(web_paths=[sages_url, ame_url])
web_docs = web_loader.load()

### Load PDF Document (ERAS)

Similar to the *WebBaseLoader*, *PyPDFLoader* is LangChain’s built-in tool for working with PDF files. I used it to process the ERAS Cholecystectomy booklet.

The main advantage is how it handles document structure. Instead of loading the entire PDF as one large block of text, it automatically splits the content page by page and includes the page number in the metadata for each section.

In [6]:
pdf_loader = PyPDFLoader(eras_pdf_url)
pdf_docs = pdf_loader.load()

In [7]:
len(pdf_docs)

12

### Combine all raw documents

In [8]:
all_raw_docs = web_docs + pdf_docs
print(f"Successfully loaded {len(all_raw_docs)} raw documents.")

Successfully loaded 14 raw documents.


### Chunking Strategy

__I chose a chunk_size of 1000 tokens with a chunk_overlap of 200. This is a well-balanced range for medical text. It is large enough to capture a complete concept, like the criteria for the Critical View of Safety, while still keeping enough overlap so continuous surgical steps stay connected and aren’t split apart.__

*RecursiveCharacterTextSplitter* is a specific chunking method in LangChain, and it’s one of the most effective options for working with standard text. Instead of splitting the text at fixed intervals, like every 500 words, which can break sentences or surgical steps, it uses a step-by-step approach.

I chose this method because clinical guidelines are highly structured, with headings, bullet points, and ordered steps. The key advantage is that it follows the natural structure of the human language. This helps ensure that important details, like safety warnings, stay connected to the surgical steps they relate to.

In [9]:
# Using RecursiveCharacterTextSplitter to split by paragraphs/sentences to preserve clinical context boundaries.

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, # simply the maximum number of characters in each text segment.
        chunk_overlap=200, # defines how much text is shared between neighboring segments.
        add_start_index=True, # Helps track where in the document the chunk came from
        separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)

chunked_docs = text_splitter.split_documents(all_raw_docs)
    
print(f"Split documents into {len(chunked_docs)} manageable chunks.")

Split documents into 274 manageable chunks.


In [12]:
print("\n--- Preview of Chunk 1 ---")
print(f"Source: {chunked_docs[2].metadata.get('source')}")
print(f"Content: {chunked_docs[2].page_content[:300]}...\n")


--- Preview of Chunk 1 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content: Member Spotlight
Give the Gift of SAGES Membership


Patients

Join the SAGES Patient Partner Network (PPN)
Patient Information Brochures
Healthy Sooner – Patient Information for Minimally Invasive Surgery
Choosing Wisely – An Initiative of the ABIM Foundation
All in the Recovery: Colorectal Cancer ...



__Unified Pipeline: It seamlessly handles both standard web scraping and PDF parsing in one go.__

__Metadata Preservation: The LangChain loaders automatically attach the source URLs to the metadata of each chunk. This is critical for RAG, as my agent will eventually need to cite exactly which document it pulled the surgical step from.__

Now that we have our text divided into overlapping clinical chunks, I need to embed them into a Vector Store so the model can query them.

# Phase 3: The RAG Pipeline (Embedding & Retrieval)

Standard top-k retrieval might pull four chunks that all describe the exact same sentence from different angles. MMR fetches a larger pool of chunks and then selects the most diverse set, ensuring the LLM gets a broader context of the surgical procedure (e.g., pulling a chunk about Calot's triangle dissection and a chunk about the safety risks).

For the embedding model, `BAAI/bge-small-en-v1.5` (via Hugging Face) is the best lightweight embedding model for retrieval tasks. Since I’m running everything locally, I don’t need a large model to achieve accurate semantic search. It has excellent semantic understanding of __medical texts__ and runs fast that the user experiences near real-time performance. For the vector database, ChromaDB is perfect because it stores the database locally as a file, requiring no complex server setup for the live demo.

## 3.1 Initializing Local Embeddings
To maintain strict data privacy suitable for hospital environments, we utilize a local, open-weights embedding model (`BAAI/bge-small-en-v1.5`). We are accelerating this process using GPU compute.

In [10]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [11]:
print("Initializing local HuggingFace embeddings...")

# Initialize Local Embeddings
# Pushing the model to the GPU for near-instantaneous embedding generation
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda:0'} 
)

print("Embeddings loaded successfully on GPU.")

Initializing local HuggingFace embeddings...
Embeddings loaded successfully on GPU.


## 3.2 Building the Vector Database

After converting sections of the SAGES and ERAS guidelines into numerical vectors, we need an efficient system to store and search them. When a user asks a question, we convert that question into a vector as well, and the database compares it to all stored vectors using cosine similarity, which measures how close they are in meaning. It then returns the most relevant sections based on that similarity.

We use ChromaDB to store our chunked clinical guidelines locally because it’s lightweight and works well for a privacy-focused setup. In a clinical setting, I wanted to avoid cloud-based vector databases, since sending hospital data to external servers conflicts with a local, secure design. The database is persisted to disk so it can be instantly reloaded in future sessions without re-embedding.

In [12]:
persist_dir = "./chroma_db_clinical"
print("Building and persisting local Vector Store. This may take a moment...")

vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    persist_directory=persist_dir
)

print(f"Successfully embedded chunks into local ChromaDB at {persist_dir}.")

Building and persisting local Vector Store. This may take a moment...
Successfully embedded chunks into local ChromaDB at ./chroma_db_clinical.


### 3.2.1 Mount the Persisted Database

In [6]:
# Point Chroma to the folder where the vectors live.
vector_store = Chroma(
    persist_directory="./chroma_db_clinical",
    embedding_function=embeddings
)

/scratch/f0034wq/ipykernel_2384889/266514095.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


## 3.3 Configuring the Clinical Retriever

Standard retrieval often pulls highly redundant chunks. To ensure our agent receives a comprehensive view of the surgical step and associated safety risks, we configure the retriever to use Maximal Marginal Relevance (MMR). MMR selects results that are closely related to the query, but it penalizes them if they are too similar to the chunks it has already selected. I used this approach for the clinical retriever because surgeons need comprehensive context, not repeated information.

In [13]:
# Configure the Retriever with MMR
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,         # Return exactly 4 distinct chunks to the LLM
        "fetch_k": 15   # Initially fetch 15 similar chunks to evaluate for diversity
    }
)

print("Retriever configured with MMR (k=4, fetch_k=15).")

Retriever configured with MMR (k=4, fetch_k=15).


When the user asks a question, first, it pushes the user's question through the `BGE-small` embedding model to turn it into a vector. Then, the __Retriever__ steps in. It scans the ChromaDB vector database and calculates the mathematical distance between the question's vector and the vectors of all our document chunks. I specifically configured it to use __MMR__ to retrieve the top 4 most relevant, yet diverse, chunks of clinical guidelines.

## 3.4 Testing the Retrieval Pipeline
Let's verify the pipeline retrieves accurate and diverse context for a critical safety concept in laparoscopic cholecystectomy.

In [15]:
# Test query relevant to the clinical guidelines
test_query = "What is the Critical View of Safety (CVS)?"
print(f"Querying Vector Store: '{test_query}'\n")

retrieved_docs = retriever.invoke(test_query)

# Display the source and content of the top retrieved chunks
for i, doc in enumerate(retrieved_docs):
    print(f"--- Retrieved Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Content snippet: {doc.page_content[:300]}...\n")

Querying Vector Store: 'What is the Critical View of Safety (CVS)?'

--- Retrieved Chunk 1 ---
Source: https://ales.amegroups.org/article/view/5766/html
Content snippet: Step 2: Establishing the critical view of safety...

--- Retrieved Chunk 2 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content snippet: GUIDELINE RECOMMENDATIONS:
Question 1: Should the critical view of safety (CVS) versus other techniques (e.g. infundibular, top down, or intraoperative cholangiography) be used to mitigate the risk of bile duct injury during laparoscopic cholecystectomy?
Recommendation: In patients undergoing laparo...

--- Retrieved Chunk 3 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content snippet: Narrative synthesis: Forty-five full text articles identified by the search methodology were reviewed that included three systematic reviews.
Use of Critical View of Sa

# Phase 4: Agent Construction & Structured Reasoning

## 4.1 Loading the Inference Engine (vLLM)
Instead of relying on external APIs, we load Llama-3 8B directly into the V100 GPU's memory using vLLM. This ensures maximum privacy for clinical environments while providing high-throughput token generation.

In [15]:
from langchain_community.llms import VLLM

project_path = "/dartfs/rc/nosnapshots/V/VaickusL-nb/EDIT_Students/users/JiQing/LLM Project"

print("Loading vLLM directly into notebook memory. This will allocate GPU VRAM...")

# Initialize vLLM inline. 
# We limit gpu_memory_utilization to 0.8, so it leaves room for our Chroma embeddings.
llm = VLLM(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    trust_remote_code=True,  # Required for some modern HuggingFace models
    max_new_tokens=1024,
    temperature=0.0,         # Zero temperature for clinical accuracy
    vllm_kwargs={"gpu_memory_utilization": 0.8},
    download_dir = f"{project_path}/cache"
)

print("\nvLLM engine loaded successfully!")

Loading vLLM directly into notebook memory. This will allocate GPU VRAM...
INFO 04-30 11:33:01 [utils.py:253] non-default args: {'trust_remote_code': True, 'download_dir': '/dartfs/rc/nosnapshots/V/VaickusL-nb/EDIT_Students/users/JiQing/LLM Project/cache', 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'meta-llama/Meta-Llama-3-8B-Instruct'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 04-30 11:33:02 [model.py:631] Resolved architecture: LlamaForCausalLM
WARNING 04-30 11:33:02 [model.py:1921] Your device 'Tesla V100-SXM2-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 04-30 11:33:02 [model.py:1971] Casting torch.bfloat16 to torch.float16.
INFO 04-30 11:33:02 [model.py:1745] Using max model len 8192
INFO 04-30 11:33:05 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-30 11:33:06 [system_utils.py:103] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=2466752) INFO 04-30 11:33:51 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='meta-llama/Meta-Llama-3-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-L

(EngineCore_DP0 pid=2466752) /dartfs/rc/nosnapshots/V/VaickusL-nb/EDIT_Students/users/JiQing/anaconda3/envs/python_3_11/lib/python3.11/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=2466752) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=2466752)   warnings.warn(


(EngineCore_DP0 pid=2466752) INFO 04-30 11:35:58 [cuda.py:418] Valid backends: ['TRITON_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=2466752) INFO 04-30 11:35:58 [cuda.py:427] Using TRITON_ATTN backend.


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:04,  1.41s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:02<00:02,  1.44s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:03<00:00,  1.04it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:04<00:00,  1.13s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:04<00:00,  1.16s/it]
(EngineCore_DP0 pid=2466752) 


(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:04 [default_loader.py:314] Loading weights took 4.75 seconds
(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:05 [gpu_model_runner.py:3338] Model loading took 14.9596 GiB memory and 117.512048 seconds
(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:38 [backends.py:631] Using cache directory: /dartfs-hpc/rc/home/q/f0034wq/.cache/vllm/torch_compile_cache/e798a1794f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:38 [backends.py:647] Dynamo bytecode transform time: 32.93 s
(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:43 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 4.572 s
(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:44 [monitor.py:34] torch.compile takes 37.50 s in total
(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:46 [gpu_worker.py:359] Available KV cache memory: 9.16 GiB
(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:46 [kv_cache_utils.py:1229] GPU KV cache

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 12.92it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:05<00:00,  6.62it/s]


(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:56 [gpu_model_runner.py:4244] Graph capturing finished in 10 secs, took 0.50 GiB
(EngineCore_DP0 pid=2466752) INFO 04-30 11:36:56 [core.py:250] init engine (profile, create kv cache, warmup model) took 51.38 seconds
INFO 04-30 11:36:58 [llm.py:352] Supported tasks: ['generate']

vLLM engine loaded successfully!


## 4.2 Defining the Structured Output Schema
To satisfy the requirement for structured reasoning, we define a Pydantic schema.

Instead of letting the LLM generate a free-form paragraph, I used Pydantic and LangChain's `JsonOutputParser`. This forces the model to map its clinical reasoning into a rigid, deterministic JSON object.

This ensures the LLM's response always breaks down the clinical scenario into exact, machine-readable fields (ex. Current Step, Next Action, Safety Warnings).

We will use *Optional* types and add a *query_intent* field. This forces the LLM to pause and categorize the question before generating the rest of the JSON.

In [16]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

# Define a flexible, intent-driven structure
class SurgicalQAOutput(BaseModel):
    query_intent: str = Field(description="Classify the query into one of four categories: 'Surgical Step Understanding', 'Clinical Reasoning', 'Knowledge Retrieval', or 'Context-based'.")
    primary_answer: str = Field(description="A detailed explanation answering the core question, grounded ONLY in retrieved guidelines.")
    current_step: Optional[str] = Field(default=None, description="The current surgical step. ONLY output this if the query involves a specific point in the surgery. Otherwise, output null.")
    next_action: Optional[str] = Field(default=None, description="The immediate next step. ONLY output this if applicable to the query. Otherwise, output null.")
    safety_warnings: Optional[List[str]] = Field(default=None, description="Key risks or safety considerations. ONLY output if applicable to the query. Otherwise, output null.")

# Initialize the parser
output_parser = JsonOutputParser(pydantic_object=SurgicalQAOutput)

print("Flexible Pydantic schema and JSON parser initialized.")

Flexible Pydantic schema and JSON parser initialized.


## 4.3 Building the RAG Generation Pipeline
We now link our Phase 3 Retriever to our Phase 4 LLM. The prompt dynamically injects the retrieved clinical guidelines, the user's question, and the JSON formatting instructions.

In addition to structured output, the system uses a __Multi-step reasoning__ process. Early on, using a fixed schema led the model to generate incorrect surgical steps for general knowledge questions.

To address this, I designed an intent-aware prompt that guides the model through a step-by-step process. __Step 1:__ It reviews the user’s question and identifies the clinical intent. __Step 2:__ Decide which fields of the JSON structure are relevant based on that classification. __Step 3:__ It generates the final response.

This step-by-step approach helps prevent incorrect or misleading structured outputs.

In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Define the Intent-Aware Prompt
prompt = PromptTemplate(
    template="""You are an expert AI clinical assistant specializing in laparoscopic cholecystectomy. 
    Answer the user's question using ONLY the provided clinical context.
    
    CRITICAL INSTRUCTIONS:
    1. First, classify the user's question type (e.g., Knowledge Retrieval vs. Surgical Step).
    2. Provide the main answer in the 'primary_answer' field.
    3. ONLY fill out 'current_step', 'next_action', and 'safety_warnings' if the question implies a specific point in the surgical workflow (e.g., dissecting Calot's triangle). 
    4. If the question is a general definition or protocol (like ERAS guidelines or CVS), you MUST set 'current_step', 'next_action', and 'safety_warnings' to null.
    
    Clinical Context:
    {context}
    
    User Question: {question}
    
    {format_instructions}
    
    Output purely the JSON object without any markdown wrapping or additional text.
    """,
    input_variables=["question", "context"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)

# Helper function to format the retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the LCEL Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

print("Intent-aware RAG Pipeline linked and ready for inference.")

Intent-aware RAG Pipeline linked and ready for inference.


Retrieving the documents is just the first step; the system also needs to use them correctly without generating incorrect information. I designed the LangChain Expression Language chain pipeline to pass the retrieved content directly into the prompt.

In the `rag_chain`, the `retriever` collects four chunks of text, and the `format_docs` helper function combines them into a single string. That string is then inserted into the `{context}` field of the `PromptTemplate`.

Because I clearly instruct the Llama-3 model to answer using only the provided clinical context, it relies on those retrieved guidelines as its main reference when reasoning through the `{question}` and completely bypasses its own pre-trained biases before generating the final JSON output.

## 4.4 Live Inference Test
Let's test the system with a complex, pseudo-visual reasoning question from the assignment prompt.

### Creating an assistant function

In [17]:
import json

def Clinical_Agent():
    print("=====================================================")
    print("⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized")
    print("Type 'exit' or 'quit' to end the session.")
    print("=====================================================\n")

    # Start an interactive CLI loop
    while True:
        # 1. Get user input
        user_input = input("\n🧑‍⚕️ Surgeon (You): ")
    
        # 2. Check for exit commands
        if user_input.lower() in ['exit', 'quit']:
            print("\nEnding session. Goodbye!")
            break
        
        # Skip empty inputs
        if not user_input.strip():
            continue
        
        print("\n🤖 AI Assistant is thinking and searching guidelines...")
    
        try:
            # 3. Invoke the RAG chain with the user's input
            result = rag_chain.invoke(user_input)
        
            # 4. Print the structured output beautifully
            print("\n--- Structured Clinical Output ---")
            print(json.dumps(result, indent=4))
        
        except Exception as e:
            print(f"\n❌ An error occurred: {e}")

### Question Category: Surgical Step Understanding

In [21]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the current step if the surgeon is dissecting Calot’s triangle?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The current step is the dissection of the hepatocystic triangle.",
    "current_step": "Dissection of the hepatocystic triangle",
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [22]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the next step after identifying the cystic duct and artery?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "After identifying the cystic duct and artery, the next step is to clip the cystic artery and divide it using hook scissors, taking care not to dislodge the proximal clips.",
    "current_step": "Step 3: Cystic artery is clipped and divided",
    "next_action": "Division of the cystic duct",
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Clinical Reasoning

In [23]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the key safety considerations during cholecystectomy?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "The key safety considerations during cholecystectomy include identifying the critical view of safety, maintaining a clear dissection plane, and avoiding excessive retraction. Additionally, surgeons should be aware of the risk of bile duct injury and take steps to minimize it, such as using a laparoscopic cholecystectomy technique that emphasizes the principles of safe cholecystectomy as highlighted by the Society of American Gastrointestinal and Endoscopic Surgeons (SAGES).",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [25]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the risks at the stage of cystic duct dissection?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The risks at the stage of cystic duct dissection are not explicitly mentioned in the provided clinical context. However, it is essential to identify and tape ligate the cystic duct prior to fundus-first dissection of the gallbladder to minimize the risk of bile duct injury.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Knowledge Retrieval

In [26]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the Critical View of Safety (CVS)?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "In patients undergoing laparoscopic cholecystectomy, the Critical View of Safety (CVS) is a technique used for anatomic identification of the cystic duct and artery.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [27]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the ERAS recommendations for cholecystectomy?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "The ERAS recommendations for cholecystectomy include a multidisciplinary approach to patient care, with a focus on minimizing postoperative complications and improving patient outcomes. This includes preoperative optimization, intraoperative techniques, and postoperative care protocols.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Context-based Question (Pseudo Visual Input)

In [19]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  The gallbladder is retracted superiorly, and dissection is being performed around Calot’s triangle. What is the current step, what is the next step, and what are the risks?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                        | 0/1 [00:00<?…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The dissection begins by incising peritoneum along the edge of the gallbladder on both sides to open up the hepatocystic triangle.",
    "current_step": "Dissecting Calot's triangle",
    "next_action": "Continue dissecting the triangle to expose the cystic duct and artery",
    "safety_warnings": "Avoid energy use near the duodenum which can be adherent to the gallbladder"
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### (Optional) Interactive UI Dashboard (`ipywidgets`)

As an alternative to the programmatic testing shown above, I have also engineered a lightweight, graphical User Interface directly within this Jupyter environment. 

While setting up a separate web framework is common in production, it adds extra network layers and delay, which aren’t needed for a local HPC demo.

To fulfill the UI requirement efficiently, this cell utilizes `ipywidgets` to generate a native, interactive dashboard. It allows us to dynamically test new clinical scenarios, ask follow-up reasoning questions, and evaluate the agent's intent-classification in real-time, completely bypassing the need to modify code or re-execute cells.

__`ipywidgets`__ allows you to build a sleek, interactive graphical interface directly inside the notebook cell.

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
def Clinical_Agent():
    # ==========================================
    # 1. Define the UI Components
    # ==========================================
    header = widgets.HTML("<h3>⚕️ Laparoscopic Cholecystectomy AI Assistant</h3><p>Enter your clinical query below to search the SAGES and ERAS guidelines.</p>")

    query_input = widgets.Textarea(
        value='',
        placeholder='e.g., "What is the current step if the surgeon is dissecting Calot’s triangle?"',
        description='🧑‍⚕️ Query:',
        layout=widgets.Layout(width='90%', height='80px')
    )

    submit_button = widgets.Button(
        description=' Ask Assistant',
        button_style='primary', # Makes the button blue
        icon='stethoscope',     # Adds a medical icon
        layout=widgets.Layout(width='200px', margin='10px 0px 10px 100px')
    )

    output_area = widgets.Output(layout=widgets.Layout(border='1px solid #d3d3d3', padding='10px', width='90%'))

    # ==========================================
    # 2. Define the Execution Logic
    # ==========================================
    def on_submit_clicked(b):
        with output_area:
            # Clear the previous output before showing the new one
            clear_output(wait=True)
        
            user_query = query_input.value.strip()
            if not user_query:
                print("⚠️ Please enter a valid question.")
                return

            print("🤖 AI Assistant is classifying intent and searching clinical guidelines...")
            
            try:
                # Invoke your Intent-Aware RAG chain
                result = rag_chain.invoke(user_query)
            
                # Print the structured output beautifully
                print("\n✅ Response Generated:\n")
                print(json.dumps(result, indent=4))
            
            except Exception as e:
                print(f"\n❌ An error occurred: {e}")

    # ==========================================
    # 3. Link and Display the UI
    # ==========================================
    submit_button.on_click(on_submit_clicked)

    # Display the elements vertically
    ui_layout = widgets.VBox([header, query_input, submit_button, output_area])
    display(ui_layout)

# Phase 5 Evaluation Script (Using RAGAS)

Getting the ground_truths is widely considered the biggest bottleneck in deploying enterprise AI. It is referred to as the "Cold Start Problem" of RAG evaluation.

In a real clinical production setting, building `ground_truths` data usually follows two main approaches.

The first is the gold-standard manual approach. We work closely with experienced surgeons and ask them to write a set of complex questions, along with the exact answers they expect the system to provide. This process takes time and resources, but it provides a reliable reference for safety and quality.

The second approach is more scalable and uses synthetic data generation. I use an LLM to create the test set. For example, I can loop through our ChromaDB, feed sections of the SAGES guidelines into Llama-3, and instruct it to act like a medical professor creating exam questions: ‘Read this paragraph and generate three factual questions with their answers.’

This allows us to quickly build a large dataset of ground-truth QA pairs, which we can then use for evaluation with frameworks like RAGAS.

## 5.1 Synthetic Test Data Generation

Since I already set up a Pydantic schema and a LangChain pipeline for my main agent, we can use that exact same architecture to build a Synthetic Data Generator.

In [18]:
import json
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

print("Initializing Synthetic Ground Truth Generator...")

# 1. Define the Pydantic Schema for the Exam
# We force the LLM to output a clean Question and Answer pair.
class QA_Pair(BaseModel):
    question: str = Field(description="A specific clinical question based strictly on the text.")
    ground_truth: str = Field(description="The exact, factual textbook answer found in the text.")

qa_parser = JsonOutputParser(pydantic_object=QA_Pair)

# 2. Build the Professor Prompt
generator_prompt = PromptTemplate(
    template="""You are an expert surgical educator creating an exam. 
    Read the following clinical guideline chunk. 
    Generate ONE difficult, specific question based on this text, and provide the exact factual answer.
    Do not use outside knowledge. 
    
    Clinical Guideline Text:
    {text_chunk}
    
    {format_instructions}
    """,
    input_variables=["text_chunk"],
    partial_variables={"format_instructions": qa_parser.get_format_instructions()},
)

# 3. Create the Generation Chain
# We reuse the `llm` (vLLM) already initialized in Phase 4
generation_chain = generator_prompt | llm | qa_parser

# 4. Generate the Synthetic Dataset
print("Generating test questions from clinical guidelines...\n")

synthetic_dataset = []

# Fetch a few random chunks from your existing vector database to test with
sample_docs = vector_store.similarity_search("Critical View of Safety", k=3)


# Loop through the chunks and have the LLM write a test for each one
for i, doc in enumerate(sample_docs):
    try:
        # Pass the raw text of the chunk to the LLM
        qa_result = generation_chain.invoke({"text_chunk": doc.page_content})
        synthetic_dataset.append(qa_result)
        
        print(f"--- Generated QA Pair #{i+1} ---")
        print(json.dumps(qa_result, indent=4))
        print("\n")
        
    except Exception as e:
        print(f"Generation failed for chunk {i}: {e}")

print("Synthetic Ground Truth dataset successfully generated. Ready for RAGAS evaluation.")

Initializing Synthetic Ground Truth Generator...
Generating test questions from clinical guidelines...



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                            | 0/1 [00:…

Generation failed for chunk 0: Invalid json output: Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the 

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                            | 0/1 [00:…

Generation failed for chunk 1: Invalid json output: Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the 

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                            | 0/1 [00:…

Generation failed for chunk 2: Invalid json output: Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the JSON output:
{"question": "What is the term used to describe the step in laparoscopic cholecystectomy where the cystic artery and cystic duct are separated from the common hepatic duct?", "ground_truth": "critical view of safety"} 





Please provide the JSON output in the exact format required. 

Here is the 

To evaluate the system, I needed a dataset of ground truths. Rather than manually copying and pasting from the ERAS PDF, I engineered an automated pipeline that inverses the RAG process. I used Llama-3 and LangChain to read the database and synthetically generate its own clinical exam. This output can be directly plugged into the RAGAS evaluation script.

## 5.2 Evaluation

Because the agent outputs dynamic JSON and free-text, you can't just use traditional machine learning metrics like F1-scores or standard accuracy.

If I were moving this pipeline from a take-home assignment into a real production environment, I would implement an evaluation framework like RAGAS (Retrieval Augmented Generation Assessment)

To truly know if the agent is performing well, you have to split the evaluation into two distinct parts:

- __Retrieval Evaluation:__ First, we have to measure if the ChromaDB is actually pulling the right clinical guidelines. We measure things like Context Precision (are the retrieved chunks highly relevant?) and Context Recall (did we miss any crucial steps from the ERAS manual?).
- __Generation Evaluation:__ Second, we evaluate the Llama-3 model’s output. The two most important metrics here are Faithfulness and Answer Relevance. Faithfulness checks that the model isn’t making things up, and that every claim in the JSON response is supported by the retrieved clinical context. Answer Relevance ensures the model is directly addressing the surgeon’s question, rather than drifting into unrelated details.

In a modern MLOps (Machine Learning Operations) pipeline, we evaluate these metrics using the ‘LLM-as-a-judge’ approach. We pass the user’s question, the retrieved context, and the model’s final JSON output to a more advanced model like GPT-4o or Claude 3.5 Sonnet, and ask it to score the response based on specific RAGAS metrics.
This allows us to run automated regression tests whenever we update the surgical guidelines or make small adjustments to the embedding model.

In [15]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

# 1. Prepare the Evaluation Dataset
# RAGAS requires 4 specific data points for every test case:
# - question: The user's query
# - answer: The actual text generated by your Llama-3 agent
# - contexts: The text chunks retrieved by ChromaDB
# - ground_truth: The "perfect" answer (used to test recall)

eval_data = {
    "question": [
        "What are the risks of dissecting Calot’s triangle?"
    ],
    "answer": [
        "The primary risk during dissection of Calot's triangle is bile duct injury..." # agent's JSON primary_answer
    ],
    "contexts": [
        [
            "Dissection of Calot's triangle must be performed carefully to avoid...", 
            "Bile duct injury is a severe complication occurring in..."
        ] # The chunks returned by your retriever
    ],
    "ground_truths": [
        ["Bile duct injury, vascular injury, and improper identification of the cystic duct."] # The textbook answer
    ]
}

# Convert the dictionary to a HuggingFace Dataset object
dataset = Dataset.from_dict(eval_data)

# 2. Run the LLM-as-a-Judge Evaluation
print("Running RAGAS Evaluation Pipeline...")
# Note: In a production setting, you can configure RAGAS to use a strong local judge (like Llama-3 70B) or OpenAI's GPT-4o.
result = evaluate(
    dataset = dataset, 
    metrics = [
        context_precision, # Did the retriever pull relevant docs?
        context_recall,    # Did the retriever miss any textbook facts?
        faithfulness,      # Did the agent hallucinate outside the context?
        answer_relevancy   # Did the agent actually answer the specific question?
    ],
)

# 3. Output the Metrics Dashboard
print("\n Evaluation Complete. Generating Report:\n")

# Convert the results to a pandas dataframe for a clean visual output
df_results = result.to_pandas()
display(df_results)

/scratch/f0034wq/ipykernel_2384889/2403358058.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/scratch/f0034wq/ipykernel_2384889/2403358058.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/scratch/f0034wq/ipykernel_2384889/2403358058.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/scratch/f0034wq/ipykernel_2384889/2403358058.py:4: DeprecationWarning: Importing context_r